# Atlas Forge Engine 0.4 API — T4 GPU

Motor configurado para Google Colab com GPU T4. Execute as células na ordem.

A etapa 5 usa Cloudflare Tunnel, sem pyngrok e sem token.


## 1. Confirmar GPU T4


In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Sessão sem GPU. Selecione Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU.')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True, check=True)
gpu_info = gpu.stdout.strip()
print('GPU disponível:', gpu_info)
if 'T4' not in gpu_info.upper():
    raise RuntimeError(f'GPU atual não é T4: {gpu_info}')
print('✅ T4 confirmada. Pode seguir para a etapa 2.')


## 2. Preparar o motor isolado


In [ ]:
import pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/content/TripoSR')
ENV = pathlib.Path('/content/atlas_forge_env')
JOBS_ROOT = pathlib.Path('/content/atlas_jobs')
JOBS_ROOT.mkdir(parents=True, exist_ok=True)
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/VAST-AI-Research/TripoSR.git',str(ROOT)], check=True)
if ENV.exists(): shutil.rmtree(ENV)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','virtualenv'], check=True)
subprocess.run([sys.executable,'-m','virtualenv','--system-site-packages',str(ENV)], check=True)
PYTHON = str(ENV/'bin'/'python')
subprocess.run([PYTHON,'-m','pip','install','-q','--upgrade','pip','setuptools','wheel'], check=True)
subprocess.run([PYTHON,'-m','pip','install','-q','--ignore-installed','numpy==1.26.4','cupy-cuda12x==13.3.0','onnxruntime==1.20.1','rembg==2.0.61','omegaconf==2.3.0','Pillow==10.1.0','einops==0.7.0','transformers==4.35.0','trimesh==4.0.5','huggingface-hub==0.17.3','imageio[ffmpeg]','xatlas==0.0.9','moderngl==5.10.0','pygltflib','fastapi','uvicorn[standard]','python-multipart'], check=True)
subprocess.run([PYTHON,'-m','pip','install','--no-build-isolation','git+https://github.com/tatsy/torchmcubes.git@3aef8afa5f21b113afc4f4ea148baee850cbd472'], check=True)
print('✅ Atlas Forge Engine preparado.')


## 3. Criar a API do motor


In [ ]:
from pathlib import Path
APP_FILE = Path('/content/atlas_forge_api.py')
APP_FILE.write_text(r'''
import json, os, shutil, subprocess, threading, uuid
from pathlib import Path
from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
ROOT=Path('/content/TripoSR'); PYTHON='/content/atlas_forge_env/bin/python'; JOBS=Path('/content/atlas_jobs'); JOBS.mkdir(exist_ok=True)
app=FastAPI(title='Atlas Forge Engine API',version='0.4')
app.add_middleware(CORSMiddleware,allow_origins=['*'],allow_credentials=False,allow_methods=['*'],allow_headers=['*'])
def save(d,p): (d/'status.json').write_text(json.dumps(p,ensure_ascii=False),encoding='utf-8')
def load(job):
 p=JOBS/job/'status.json'
 if not p.exists(): raise HTTPException(404,'job não encontrado')
 return json.loads(p.read_text(encoding='utf-8'))
def patch():
 p=ROOT/'tsr'/'bake_texture.py'; c=p.read_text(encoding='utf-8')
 c=c.replace('moderngl.create_context(standalone=True)',"moderngl.create_context(standalone=True, backend='egl')")
 c=c.replace('positions = torch.tensor(positions_texture.reshape(-1, 4)[:, :-1])','positions = torch.tensor(positions_texture.reshape(-1, 4)[:, :-1], device=scene_code.device)')
 c=c.replace('rgb_f = queried_grid[\"color\"].numpy().reshape(-1, 3)','rgb_f = queried_grid[\"color\"].detach().cpu().numpy().reshape(-1, 3)')
 p.write_text(c,encoding='utf-8')
def worker(job,inp,quality,bg):
 d=JOBS/job; out=d/'output'
 try:
  save(d,{'jobId':job,'status':'processing','progress':20,'message':'Gerando modelo 3D...','modelUrl':None}); patch()
  if out.exists(): shutil.rmtree(out)
  out.mkdir(); tex={'fast':'512','balanced':'1024','high':'2048'}.get(quality,'1024')
  cmd=[PYTHON,'run.py',inp,'--output-dir',str(out),'--bake-texture','--texture-resolution',tex]
  if bg: cmd.append('--remove-bg')
  env=os.environ.copy(); env['PYOPENGL_PLATFORM']='egl'
  r=subprocess.run(cmd,cwd=ROOT,capture_output=True,text=True,env=env)
  if r.returncode: raise RuntimeError(r.stderr or 'Falha no TripoSR')
  meshes=list(out.rglob('*.glb'))+list(out.rglob('*.obj'))
  if not meshes: raise RuntimeError('Nenhuma malha encontrada')
  target=d/'model.glb'; src=meshes[0]
  if src.suffix.lower()=='.glb': shutil.copy2(src,target)
  else:
   code=f"import trimesh; s=trimesh.load(r'{src}',force='scene',process=False); s.export(r'{target}',file_type='glb')"
   c=subprocess.run([PYTHON,'-c',code],capture_output=True,text=True)
   if c.returncode: raise RuntimeError(c.stderr)
  u=f'/models/{job}'; save(d,{'jobId':job,'status':'done','progress':100,'message':'Modelo pronto','modelUrl':u,'glbUrl':u,'url':u})
 except Exception as e: save(d,{'jobId':job,'status':'error','progress':100,'message':str(e),'modelUrl':None})
@app.get('/')
def root(): return {'ok':True,'service':'atlas-forge-engine-api','version':'0.4'}
@app.post('/image-to-3d')
async def create(image:UploadFile=File(...),quality:str=Form('balanced'),removeBackground:str=Form('true'),prompt:str=Form('')):
 suf=Path(image.filename or 'upload.png').suffix.lower()
 if suf not in {'.png','.jpg','.jpeg','.webp'}: raise HTTPException(400,'Use PNG, JPG ou WEBP')
 job=uuid.uuid4().hex[:12]; d=JOBS/job; d.mkdir(); inp=d/f'input{suf}'; inp.write_bytes(await image.read())
 save(d,{'jobId':job,'status':'queued','progress':5,'message':'Job criado','modelUrl':None})
 threading.Thread(target=worker,args=(job,str(inp),quality,str(removeBackground).lower()!='false'),daemon=True).start()
 return {'jobId':job,'status':'queued','statusUrl':f'/image-to-3d-status/{job}'}
@app.get('/image-to-3d-status/{job}')
def status(job:str): return load(job)
@app.get('/models/{job}')
def model(job:str):
 p=JOBS/job/'model.glb'
 if not p.exists(): raise HTTPException(404,'Modelo ainda não disponível')
 return FileResponse(p,media_type='model/gltf-binary',filename=f'{job}.glb')
''',encoding='utf-8')
print('✅ API criada:', APP_FILE)


## 4. Subir o servidor


In [ ]:
import subprocess, time, requests
try:
    server.terminate()
except Exception:
    pass
server=subprocess.Popen([PYTHON,'-m','uvicorn','atlas_forge_api:app','--host','0.0.0.0','--port','8000'],cwd='/content')
time.sleep(5)
r=requests.get('http://127.0.0.1:8000',timeout=10)
print('✅ Servidor ativo:',r.json())


## 5. Gerar URL pública com Cloudflare Tunnel


In [ ]:
import os, re, subprocess, time, pathlib
CLOUDFLARED=pathlib.Path('/content/cloudflared')
if not CLOUDFLARED.exists():
    subprocess.run(['wget','-q','-O',str(CLOUDFLARED),'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],check=True)
    CLOUDFLARED.chmod(0o755)
try:
    tunnel.terminate()
except Exception:
    pass
log_path='/content/cloudflared.log'
log=open(log_path,'w')
tunnel=subprocess.Popen([str(CLOUDFLARED),'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=log,stderr=subprocess.STDOUT)
public_url=None
for _ in range(40):
    time.sleep(1)
    text=pathlib.Path(log_path).read_text(errors='ignore')
    m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
    if m:
        public_url=m.group(0); break
if not public_url:
    raise RuntimeError('Não foi possível criar o túnel. Rode novamente esta célula.')
print('✅ COLE ESTA URL NO DEAD-ZONE:')
print(public_url)
